In [ ]:
# !pip install -q transformers datasets peft accelerate bitsandbytes torch

In [ ]:
# Ячейка 2: Импорты и настройка
import os
import torch
import json
from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

In [ ]:
# Ячейка 3: Конфигурация – выберите мир и пути
WORLD = "cyberpunk"          # "cyberpunk" или "medieval"
DATA_FILE = f"narrator/data/{WORLD}_train.jsonl"
OUTPUT_DIR = f"narrator/adapters/lora_{WORLD}"

BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"   # или "Qwen/Qwen2.5-7B"

# Параметры обучения
BATCH_SIZE = 2
GRAD_ACCUM = 4
EPOCHS = 3
LEARNING_RATE = 2e-4
MAX_LENGTH = 512

# Использовать квантование 4bit? (экономит VRAM)
USE_4BIT = True

In [ ]:
# Ячейка 4: Загрузка и подготовка данных
def parse_chat_from_text(text: str):
    """
    Превращает строку с тегами <|system|>, <|user|>, <|assistant|>
    в список сообщений для Qwen2.5 (ChatML).
    """
    parts = text.split("<|assistant|>")
    if len(parts) != 2:
        return None
    context_part, assistant_part = parts
    assistant_msg = assistant_part.strip()

    # Извлекаем system и user
    system_start = context_part.find("<|system|>")
    user_start = context_part.find("<|user|>")
    if system_start == -1 or user_start == -1:
        return None

    system_msg = context_part[system_start+len("<|system|>"):user_start].strip()
    user_msg = context_part[user_start+len("<|user|>"):].strip()

    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": assistant_msg}
    ]
    return messages

def load_and_prepare_data(filepath):
    data = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)
            messages = parse_chat_from_text(item["text"])
            if messages:
                data.append(messages)
    return data

train_data = load_and_prepare_data(DATA_FILE)
print(f"Загружено {len(train_data)} примеров для мира {WORLD}")

# Преобразуем в Dataset
dataset = Dataset.from_list([{"messages": msgs} for msgs in train_data])

In [ ]:
# Ячейка 5: Загрузка токенизатора и модели
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# Настройка квантования
if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model = prepare_model_for_kbit_training(model)
else:
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
        trust_remote_code=True,
    )

In [ ]:
# Ячейка 6: Функция токенизации с ChatML шаблоном Qwen
def tokenize_function(example):
    messages = example["messages"]
    # Применяем шаблон Qwen (автоматически добавляет <|im_start|> и <|im_end|>)
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    result = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )
    result["labels"] = result["input_ids"].copy()
    return result

tokenized_dataset = dataset.map(tokenize_function, batched=False)